# 5. Model Fit: Overfitting on Fashion-MNIST

Last week we built our first neural network with `PyTorch` and, in the last coding task, tracked training and test accuracy after every epoch.
This week we take that network to a harder dataset and then **deliberately make it overfit**, so we can see what overfitting looks like in the loss and accuracy curves.

We split the data into **training**, **validation** and **test** data, train on the training data, watch the validation data during training, and use the test data only once at the very end.

## Part 1: Make the network overfit

### Load Fashion-MNIST

[Fashion-MNIST](https://github.com/zalandoresearch/fashion-mnist) has the same format as MNIST: 70,000 grayscale images of 28x28 = 784 pixels, 10 classes.
But instead of digits, the images show clothing items, which makes the task considerably harder.
We load it the same way as MNIST last week (the first download takes a minute):

In [1]:
from sklearn.datasets import fetch_openml
X, y = fetch_openml('Fashion-MNIST', version=1, return_X_y=True, cache=True, parser='auto')
X.shape

(70000, 784)

The data looks exactly like MNIST, one column per pixel:

In [ ]:
X.head(3)

The label is again a number between 0-9, but now each number stands for a type of clothing:

In [ ]:
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
y.head(3)

Let's look at one example image per class:

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(10, 4.5))
for label, ax in enumerate(axes.flat):
    idx = (y.astype(int) == label).values.argmax()   # first image with this label
    ax.imshow(X.iloc[idx].values.reshape(28, 28), cmap="gray_r")
    ax.set_title(f"{label}: {class_names[label]}")
    ax.axis("off")
plt.tight_layout()

### Prepare the data: train, validation and test

We scale the pixels to [0, 1] as last week. Then we split the data into three parts:
- **Test data:** the last 10,000 images (this is the official Fashion-MNIST test set). We only use it at the very end.
- **Validation data:** 10,000 images that we evaluate after every epoch to monitor overfitting.
- **Training data:** we use only **5,000** images. This keeps training fast on a CPU, and it makes overfitting appear within a few hundred epochs. (Less training data is also one of the reasons for overfitting from the lecture.)

In [ ]:
from sklearn.model_selection import train_test_split

X_scaled = (X/255).astype('float32').values
y = y.astype('long').values

X_pool, y_pool = X_scaled[:60000], y[:60000]   # official training part
X_test, y_test = X_scaled[60000:], y[60000:]   # official test part

X_train, X_val, y_train, y_val = train_test_split(
    X_pool, y_pool, train_size=5000, test_size=10000, random_state=42, stratify=y_pool)
X_train.shape, X_val.shape, X_test.shape

As last week, we select the device and convert everything into PyTorch tensors:

In [ ]:
import torch
import torch.nn as nn
torch.manual_seed(0) # to ensure reproducibility

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

In [ ]:
train_x = torch.Tensor(X_train).float().to(device)
val_x = torch.Tensor(X_val).float().to(device)
test_x = torch.Tensor(X_test).float().to(device)
train_y = torch.Tensor(y_train).long().to(device)
val_y = torch.Tensor(y_val).long().to(device)
test_y = torch.Tensor(y_test).long().to(device)

### A reusable training function

We will train several networks, so we put last week's training loop (including the accuracy tracking from coding task 3) into a function.
After every epoch it measures loss and accuracy on the training **and** the validation data and stores them in a dictionary `history`.

Note that we evaluate in `eval()` mode and inside `torch.no_grad()`: we do not need gradients for evaluation, and some layers (e.g. dropout) behave differently during training and evaluation.

In [ ]:
loss_func = nn.CrossEntropyLoss()

def evaluate(model, x, y):
    """Returns (loss, accuracy) of the model on the data x, y."""
    model.eval()
    with torch.no_grad():
        y_hat = model(x)
        loss = loss_func(y_hat, y).item()
        accuracy = (torch.argmax(y_hat, 1) == y).float().mean().item()
    return loss, accuracy


def train(model, optimizer, no_epochs, print_every=100):
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    for epoch in range(no_epochs):

        model.train()   # evaluate() below switches to eval mode, so switch back every epoch (matters for dropout)
        optimizer.zero_grad()
        y_hat = model(train_x)            # all training points (= batch gradient descent)
        loss = loss_func(y_hat, train_y)
        loss.backward()
        optimizer.step()

        train_loss, train_acc = evaluate(model, train_x, train_y)
        val_loss, val_acc = evaluate(model, val_x, val_y)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if epoch % print_every == 0:
            print(f"Epoch {epoch}: train loss {train_loss:.3f}, val loss {val_loss:.3f}, "
                  f"train acc {train_acc:.3f}, val acc {val_acc:.3f}")
    return history

And a function that plots loss and accuracy curves next to each other. The dashed line marks the epoch with the lowest validation loss:

In [ ]:
""" You do not have to read this cell in detail """
import numpy as np

def plot_history(history, title=""):
    best_epoch = int(np.argmin(history["val_loss"]))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(history["train_loss"], label="training loss")
    ax1.plot(history["val_loss"], label="validation loss")
    ax1.set_xlabel("number of epochs"); ax1.set_ylabel("loss")
    ax2.plot(history["train_acc"], label="training accuracy")
    ax2.plot(history["val_acc"], label="validation accuracy")
    ax2.set_xlabel("number of epochs"); ax2.set_ylabel("accuracy")
    for ax in (ax1, ax2):
        ax.axvline(best_epoch, color="gray", linestyle="--", label=f"lowest val loss (epoch {best_epoch})")
        ax.legend()
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

The curves can be a bit noisy from epoch to epoch. To compare models, we therefore average the last 20 epochs:

In [ ]:
def summary(history, name=""):
    """Averages over the last 20 epochs: train acc, val acc, gap, val loss."""
    train_acc = np.mean(history["train_acc"][-20:])
    val_acc = np.mean(history["val_acc"][-20:])
    val_loss = np.mean(history["val_loss"][-20:])
    gap = train_acc - val_acc
    if name:
        print(f"{name:22s}: train acc {train_acc:.3f}, val acc {val_acc:.3f}, gap {gap:.3f}, "
              f"val loss {val_loss:.3f} (lowest: {min(history['val_loss']):.3f})")
    return train_acc, val_acc, gap, val_loss

### Step 1: Last week's network on Fashion-MNIST

We start with the network from last week. For training we now use the **Adam** optimizer (see the gradient descent variants from week 3) instead of plain SGD. We use the same optimizer settings for all networks in this notebook, so that differences between the networks come from the architecture, not from the optimizer.

### Task 1: Define last week's network
Write a class `DeepNeuralNetwork` (inheriting from `nn.Module`) with a constructor `__init__` and a `forward` method. The network has the following layers:
- Input layer of size 784
- First hidden layer of size 128 with sigmoid activation
- Second hidden layer of size 64 with sigmoid activation
- Output layer of size 10 (no activation, the softmax is part of `CrossEntropyLoss`)

In [ ]:
# check for task 1: do not change this cell
test_model = DeepNeuralNetwork()
n_params = sum(p.numel() for p in test_model.parameters())
assert n_params == 109386, f"DeepNeuralNetwork has {n_params} parameters, but should have 109,386 - check the layer sizes"
assert test_model(torch.zeros(2, 784)).shape == (2, 10), "the network should return 10 scores per image"
print("All checks passed!")

Now we create the network and train it:

In [ ]:
torch.manual_seed(0)
dnn = DeepNeuralNetwork().to(device)
print("Number of parameters:", sum(p.numel() for p in dnn.parameters()))

optimizer = torch.optim.Adam(dnn.parameters(), lr=0.0005)
history_small = train(dnn, optimizer, no_epochs=300, print_every=30)

In [ ]:
plot_history(history_small, "Last week's network")

**Question:** Is this model overfitting, underfitting, or well fitted? Look at both the gap between the curves and at how the curves are behaving at the end of training.

*Your answer:* ...

### Step 2: Extend the network until it overfits

To provoke overfitting we give the network much more **capacity** (see lecture). We make two changes:
1. **Wider and deeper:** three hidden layers with 512, 512 and 256 neurons.
2. **ReLU** instead of sigmoid in the hidden layers, so that the deeper network trains well.

### Task 2: Define the extended network
Write a class `BigNetwork` in the same way as `DeepNeuralNetwork` in Task 1, i.e. implement the constructor `__init__` and the `forward` method. The network has the following layers:
- Input layer of size 784
- First hidden layer of size 512 with ReLU activation
- Second hidden layer of size 512 with ReLU activation
- Third hidden layer of size 256 with ReLU activation
- Output layer of size 10 (no activation, as last week)

Hint: `torch.relu` works just like `torch.sigmoid`.

In [ ]:
# check for task 2: do not change this cell
test_model = BigNetwork()
n_params = sum(p.numel() for p in test_model.parameters())
assert n_params == 798474, f"BigNetwork has {n_params} parameters, but should have 798,474 - check the layer sizes"
assert test_model(torch.zeros(2, 784)).shape == (2, 10), "the network should return 10 scores per image"
print("All checks passed!")

Now we create the network and compare its size with the amount of training data:

In [ ]:
torch.manual_seed(0)
big = BigNetwork().to(device)
print("Number of parameters:", sum(p.numel() for p in big.parameters()))
print("Number of training images:", len(train_x))

The network now has about 800,000 parameters, i.e. more than 150 parameters per training image. We train it with the same settings as before (this takes about one or two minutes on a CPU):

In [ ]:
optimizer = torch.optim.Adam(big.parameters(), lr=0.0005)
history_big = train(big, optimizer, no_epochs=300, print_every=30)

In [ ]:
plot_history(history_big, "Extended network")

This is the textbook picture of overfitting:
- The **training loss** keeps decreasing towards zero and the training accuracy approaches 100%: the network starts to memorize the 5,000 training images.
- The **validation loss** reaches its minimum early (dashed line) and then *increases* again, while the validation accuracy stagnates. Compare this with the loss/capacity plots in the lecture.
- The gap between training and validation accuracy is now much larger than in Step 1.

In [ ]:
summary(history_small, "last week's network")
summary(history_big, "extended network");

**Question:** After the dashed line, the validation *loss* rises again, but the validation *accuracy* stays almost the same. How can both be true at the same time?
(Hint: cross-entropy does not only look at whether a prediction is correct, but also at how confident it is.)

*Your answer:* ...

## Part 2: Decreasing overfitting

Now it is your turn. Try out the four approaches from the lecture to decrease the overfitting of `BigNetwork`:
1. L2 regularization
2. Dropout
3. Early stopping
4. Capacity reduction

Unless stated otherwise, always start from a **fresh** model (`torch.manual_seed(0)` before creating it), use Adam with `lr=0.0005` and train for 300 epochs, so that the results are comparable with `history_big`.
Each task ends with a check cell that you should run unchanged.

### Task 3: L2 regularization
PyTorch calls L2 regularization *weight decay* and adds it via the optimizer: `torch.optim.Adam(model.parameters(), lr=0.0005, weight_decay=...)`.
1. Train a fresh `BigNetwork` with `weight_decay=0.01`. Store the history in `history_l2` and plot it.
2. Compare the plot with the one of the extended network: what happens to the gap and to the validation loss?

In [ ]:
# check for task 3: do not change this cell
_, _, gap_big, val_loss_big = summary(history_big, "no regularization")
_, _, gap_l2, val_loss_l2 = summary(history_l2, "L2 regularization")
assert gap_l2 < 0.6 * gap_big, "the gap should be clearly smaller than without regularization"
assert val_loss_l2 < val_loss_big, "the validation loss at the end should be lower than without regularization"
print("All checks passed!")

### Task 4: Dropout
1. Write a class `DropoutNetwork` that is identical to `BigNetwork`, but applies dropout with `p=0.5` after every ReLU. For this, create a layer `nn.Dropout(p=0.5)` in the constructor and call it in the `forward` method after each `torch.relu`.
2. Train it **without** weight decay. Store the trained model in `model_do`, the history in `history_dropout` and plot it.
3. How does dropout change the validation accuracy? Does it stop the validation loss from rising in our setting?

In [ ]:
# check for task 4: do not change this cell
dropout_layers = [m for m in model_do.modules() if isinstance(m, nn.Dropout)]
assert len(dropout_layers) > 0, "DropoutNetwork should contain an nn.Dropout layer"
assert all(m.p == 0.5 for m in dropout_layers), "use dropout with p=0.5"
summary(history_big, "no regularization")
summary(history_dropout, "dropout")
print("All checks passed!")

*Your answer:* ...

### Task 5: Early stopping
1. Copy the `train` function into a new function `train_early_stopping(model, optimizer, max_epochs, patience)`.
2. Whenever the validation loss reaches a new minimum, store a copy of the weights with `best_weights = copy.deepcopy(model.state_dict())` and remember the epoch in `best_epoch`.
   (Use `copy.deepcopy`: without it, the stored `state_dict()` would keep changing together with the model.)
3. Stop the training if the validation loss has not improved for `patience` epochs.
4. After the loop, restore the best weights with `model.load_state_dict(best_weights)`. The function should return `history` and `best_epoch`.
5. Train a fresh `BigNetwork` with `max_epochs=300` and `patience=20`. Store the trained model in `model_es`, the results in `history_es, best_epoch` and plot the history.

In [ ]:
# check for task 5: do not change this cell
model_val_loss, _ = evaluate(model_es, val_x, val_y)
assert best_epoch == int(np.argmin(history_es["val_loss"])), "best_epoch is not the epoch with the lowest validation loss"
assert len(history_es["val_loss"]) < 300, "training should have stopped early"
assert abs(model_val_loss - min(history_es["val_loss"])) < 1e-4, "the model does not have the best weights - did you restore them?"
print(f"Stopped after {len(history_es['val_loss'])} epochs, best epoch {best_epoch}. All checks passed!")

### Task 6: Capacity reduction
1. Write a class `SmallNetwork` with fewer parameters than `BigNetwork`: one hidden layer of 64 neurons with ReLU activation. (Tip: pass the hidden layer size as a parameter to the constructor, e.g. `def __init__(self, hidden_size=64)`, then it is easy to try other sizes.)
2. Print the number of parameters, train the network and store the history in `history_small_relu`. Plot it.
3. Try one or two other sizes. How small can the network get before it starts to underfit?

In [ ]:
# check for task 6: do not change this cell
_, _, gap_big, val_loss_big = summary(history_big, "BigNetwork")
_, _, gap_cap, val_loss_cap = summary(history_small_relu, "SmallNetwork")
assert gap_cap < 0.6 * gap_big, "the gap should be clearly smaller than with BigNetwork"
assert val_loss_cap < val_loss_big, "the validation loss at the end should be lower than with BigNetwork"
print("All checks passed!")

### Task 7: Comparison and final test
1. Run the cell below to compare all approaches.
2. Which approach works best for our data? Pick your final model, and only then evaluate it **once** on the test data with `evaluate(model, test_x, test_y)`.
   Why should we not pick the model based on the test accuracy?

In [ ]:
for name, h in [("no regularization", history_big), ("L2 regularization", history_l2),
                ("dropout", history_dropout), ("early stopping", history_es),
                ("capacity reduction", history_small_relu)]:
    summary(h, name)

*Your answer:* ...